# Wind Speed Estimation Comparison

Comparing different control scenarios with WE_Mode settings:
- **Baseline**: TCIPC_MaxTipDeflection = 25, ZeroYawDeflection = 0
- **Free yaw**: TCIPC_MaxTipDeflection = 0, ZeroYawDeflection = 0  
- **Zero yaw**: TCIPC_MaxTipDeflection = 0, ZeroYawDeflection = 1

Each scenario is run with different wind speed estimators and controller modes:
- **WE_Mode = 0** (hub), standard controller: Solid lines
- **WE_Mode = 2** (WSE), standard controller: Dashed lines
- **WE_Mode = 0** (hub), kw^2 controller: Dotted lines

In [ ]:
import glob
import os
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from weis.visualization.utils import load_OMsql_multi
from rosco.toolbox.ofTools.fast_io import output_processing

plt.style.use("journal.mplstyle")

%matplotlib widget

## Load Metadata

Load the SQL files to map ranks to design variables (TCIPC_MaxTipDeflection and TCIPC_ZeroYawDeflection).

In [ ]:
# Load metadata for WE_Mode = 0 (hub)
log_path_hub = (
    "../../../data/optimal_tuning/check_estimation_hub/log_check_estimation_hub.sql*"
)
data_dict_hub = load_OMsql_multi(log_path_hub)
df_meta_hub = pd.DataFrame(
    {
        "rank": data_dict_hub["rank"],
        "iter": data_dict_hub["iter"],
        "max_tip_deflection": data_dict_hub["tune_rosco_ivc.TCIPC_MaxTipDeflection"],
        "zero_yaw_deflection": data_dict_hub["tune_rosco_ivc.TCIPC_ZeroYawDeflection"],
    }
)
df_meta_hub["we_mode"] = 0
df_meta_hub["controller_mode"] = "standard"
print("Hub (WE_Mode=0) metadata:")
print(df_meta_hub)

# Load metadata for WE_Mode = 2 (WSE)
log_path_wse = (
    "../../../data/optimal_tuning/check_estimation_WSE/log_check_estimation_WSE.sql*"
)
data_dict_wse = load_OMsql_multi(log_path_wse)
df_meta_wse = pd.DataFrame(
    {
        "rank": data_dict_wse["rank"],
        "iter": data_dict_wse["iter"],
        "max_tip_deflection": data_dict_wse["tune_rosco_ivc.TCIPC_MaxTipDeflection"],
        "zero_yaw_deflection": data_dict_wse["tune_rosco_ivc.TCIPC_ZeroYawDeflection"],
    }
)
df_meta_wse["we_mode"] = 2
df_meta_wse["controller_mode"] = "standard"
print("\nWSE (WE_Mode=2) metadata:")
print(df_meta_wse)

# Load metadata for kw2 controller (WE_Mode = 0, VS_ControlMode = 1)
log_path_kw2 = (
    "../../../data/optimal_tuning/check_estimation_kw2/log_check_estimation_kw2.sql*"
)
data_dict_kw2 = load_OMsql_multi(log_path_kw2)
df_meta_kw2 = pd.DataFrame(
    {
        "rank": data_dict_kw2["rank"],
        "iter": data_dict_kw2["iter"],
        "max_tip_deflection": data_dict_kw2["tune_rosco_ivc.TCIPC_MaxTipDeflection"],
        "zero_yaw_deflection": data_dict_kw2["tune_rosco_ivc.TCIPC_ZeroYawDeflection"],
    }
)
df_meta_kw2["we_mode"] = 0
df_meta_kw2["controller_mode"] = "kw2"
print("\nkw2 (VS_ControlMode=1) metadata:")
print(df_meta_kw2)

# Combine metadata
df_meta = pd.concat([df_meta_hub, df_meta_wse, df_meta_kw2], ignore_index=True)
print(f"\nTotal cases: {len(df_meta)}")

## Load ROSCO Debug Files

Load the ROSCO debug files to extract wind speed estimates and power.

In [ ]:
# Load ROSCO debug files for wind speed estimation
print("Loading ROSCO debug files...")

# Hub (WE_Mode=0)
base_path_hub = "../../../data/optimal_tuning/check_estimation_hub/openfast_runs"
dbg_files_hub = sorted(glob.glob(os.path.join(base_path_hub, "rank_*/*.RO.dbg")))
op_rosco_hub = output_processing.output_processing()
rosco_data_hub = op_rosco_hub.load_fast_out(dbg_files_hub, tmin=0, verbose=False)
print(f"Loaded {len(rosco_data_hub)} ROSCO debug files for hub")

# WSE (WE_Mode=2)
base_path_wse = "../../../data/optimal_tuning/check_estimation_WSE/openfast_runs"
dbg_files_wse = sorted(glob.glob(os.path.join(base_path_wse, "rank_*/*.RO.dbg")))
op_rosco_wse = output_processing.output_processing()
rosco_data_wse = op_rosco_wse.load_fast_out(dbg_files_wse, tmin=0, verbose=False)
print(f"Loaded {len(rosco_data_wse)} ROSCO debug files for WSE")

# kw2 controller (WE_Mode=0, VS_ControlMode=1)
base_path_kw2 = "../../../data/optimal_tuning/check_estimation_kw2/openfast_runs"
dbg_files_kw2 = sorted(glob.glob(os.path.join(base_path_kw2, "rank_*/*.RO.dbg")))
op_rosco_kw2 = output_processing.output_processing()
rosco_data_kw2 = op_rosco_kw2.load_fast_out(dbg_files_kw2, tmin=0, verbose=False)
print(f"Loaded {len(rosco_data_kw2)} ROSCO debug files for kw2")

# Verify WE_Vw column exists
if "WE_Vw" in rosco_data_hub[0]["meta"]["channels"]:
    print("✓ WE_Vw (estimated wind speed) column found in debug files")
else:
    print("✗ WARNING: WE_Vw column not found!")

In [ ]:
# Load OpenFAST output files for power
print("\nLoading OpenFAST output files...")

# Hub (WE_Mode=0)
outb_files_hub = sorted(glob.glob(os.path.join(base_path_hub, "rank_*/*.outb")))
op_openfast_hub = output_processing.output_processing()
openfast_data_hub = op_openfast_hub.load_fast_out(outb_files_hub, tmin=0, verbose=False)
print(f"Loaded {len(openfast_data_hub)} OpenFAST files for hub")

# WSE (WE_Mode=2)
outb_files_wse = sorted(glob.glob(os.path.join(base_path_wse, "rank_*/*.outb")))
op_openfast_wse = output_processing.output_processing()
openfast_data_wse = op_openfast_wse.load_fast_out(outb_files_wse, tmin=0, verbose=False)
print(f"Loaded {len(openfast_data_wse)} OpenFAST files for WSE")

# kw2 controller
outb_files_kw2 = sorted(glob.glob(os.path.join(base_path_kw2, "rank_*/*.outb")))
op_openfast_kw2 = output_processing.output_processing()
openfast_data_kw2 = op_openfast_kw2.load_fast_out(outb_files_kw2, tmin=0, verbose=False)
print(f"Loaded {len(openfast_data_kw2)} OpenFAST files for kw2")

# Verify GenPwr column exists
if "GenPwr" in openfast_data_hub[0]["meta"]["channels"]:
    print("✓ GenPwr (generator power) column found in .outb files")
else:
    print("✗ WARNING: GenPwr column not found!")

## Organize Data by Scenario

Match each rank to its scenario based on design variables.

In [ ]:
# Define scenario labels based on design variables
def get_scenario_label(max_tip_deflection, zero_yaw_deflection):
    """Map design variables to scenario name."""
    if max_tip_deflection == 25 and zero_yaw_deflection == 0:
        return "Baseline"
    elif max_tip_deflection == 0 and zero_yaw_deflection == 0:
        return "Free yaw"
    elif max_tip_deflection == 0 and zero_yaw_deflection == 1:
        return "Zero yaw"
    else:
        return None  # This case will be filtered out


# Add scenario labels to metadata
df_meta["scenario"] = df_meta.apply(
    lambda row: get_scenario_label(
        row["max_tip_deflection"], row["zero_yaw_deflection"]
    ),
    axis=1,
)

# Filter out unwanted scenario (MaxTipDeflection=25, ZeroYawDeflection=1)
df_meta_filtered = df_meta[df_meta["scenario"].notna()].copy()

print("Filtered scenarios:")
print(
    df_meta_filtered[
        ["rank", "we_mode", "max_tip_deflection", "zero_yaw_deflection", "scenario"]
    ].sort_values(["scenario", "we_mode", "rank"])
)

In [ ]:
# Helper function to extract rank number from filename
def extract_rank(filename):
    """Extract rank number from filename."""
    match = re.search(r"rank_(\d+)", filename)
    return int(match.group(1)) if match else None


# Organize data into a dictionary by (scenario, we_mode, controller_mode)
data_by_scenario = {}

# Process hub data (WE_Mode=0, standard controller)
for rosco_dict, openfast_dict in zip(rosco_data_hub, openfast_data_hub):
    rank = extract_rank(rosco_dict["meta"]["filename"])
    if rank is None:
        continue

    meta_row = df_meta_filtered[
        (df_meta_filtered["rank"] == rank)
        & (df_meta_filtered["we_mode"] == 0)
        & (df_meta_filtered["controller_mode"] == "standard")
    ]
    if len(meta_row) == 0:
        continue

    scenario = meta_row["scenario"].values[0]
    we_mode = 0
    controller_mode = "standard"
    key = (scenario, we_mode, controller_mode)

    data_by_scenario[key] = {
        "time_rosco": rosco_dict["Time"],
        "time_openfast": openfast_dict["Time"],
        "wind_speed": rosco_dict["WE_Vw"],
        "power": openfast_dict["GenPwr"],
        "rotor_speed": openfast_dict["RotSpeed"],
        "rotor_speed_ref": rosco_dict["VS_RefSpd"],
        "rotor_torque": openfast_dict["RotTorq"],
        "blade_pitch": openfast_dict["BldPitch1"],
        "root_torque": openfast_dict["RootMxc1"],
        "azimuth": openfast_dict["Azimuth"],
        "scenario": scenario,
        "we_mode": we_mode,
        "controller_mode": controller_mode,
    }

# Process WSE data (WE_Mode=2, standard controller)
for rosco_dict, openfast_dict in zip(rosco_data_wse, openfast_data_wse):
    rank = extract_rank(rosco_dict["meta"]["filename"])
    if rank is None:
        continue

    meta_row = df_meta_filtered[
        (df_meta_filtered["rank"] == rank)
        & (df_meta_filtered["we_mode"] == 2)
        & (df_meta_filtered["controller_mode"] == "standard")
    ]
    if len(meta_row) == 0:
        continue

    scenario = meta_row["scenario"].values[0]
    we_mode = 2
    controller_mode = "standard"
    key = (scenario, we_mode, controller_mode)

    data_by_scenario[key] = {
        "time_rosco": rosco_dict["Time"],
        "time_openfast": openfast_dict["Time"],
        "wind_speed": rosco_dict["WE_Vw"],
        "power": openfast_dict["GenPwr"],
        "rotor_speed": openfast_dict["RotSpeed"],
        "rotor_speed_ref": rosco_dict["VS_RefSpd"],
        "rotor_torque": openfast_dict["RotTorq"],
        "blade_pitch": openfast_dict["BldPitch1"],
        "root_torque": openfast_dict["RootMxc1"],
        "azimuth": openfast_dict["Azimuth"],
        "scenario": scenario,
        "we_mode": we_mode,
        "controller_mode": controller_mode,
    }

# Process kw2 data (WE_Mode=0, kw2 controller)
for rosco_dict, openfast_dict in zip(rosco_data_kw2, openfast_data_kw2):
    rank = extract_rank(rosco_dict["meta"]["filename"])
    if rank is None:
        continue

    meta_row = df_meta_filtered[
        (df_meta_filtered["rank"] == rank)
        & (df_meta_filtered["controller_mode"] == "kw2")
    ]
    if len(meta_row) == 0:
        continue

    scenario = meta_row["scenario"].values[0]
    we_mode = 0
    controller_mode = "kw2"
    key = (scenario, we_mode, controller_mode)

    data_by_scenario[key] = {
        "time_rosco": rosco_dict["Time"],
        "time_openfast": openfast_dict["Time"],
        "wind_speed": rosco_dict["WE_Vw"],
        "power": openfast_dict["GenPwr"],
        "rotor_speed": openfast_dict["RotSpeed"],
        "rotor_speed_ref": rosco_dict["VS_RefSpd"],
        "rotor_torque": openfast_dict["RotTorq"],
        "blade_pitch": openfast_dict["BldPitch1"],
        "root_torque": openfast_dict["RootMxc1"],
        "azimuth": openfast_dict["Azimuth"],
        "scenario": scenario,
        "we_mode": we_mode,
        "controller_mode": controller_mode,
    }

print(
    f"\nOrganized data for {len(data_by_scenario)} scenario-we_mode-controller combinations:"
)
for key in sorted(data_by_scenario.keys()):
    print(f"  {key[0]}, WE_Mode={key[1]}, controller={key[2]}")

## Create Comparison Plots

7x3 subplot grid:
- **Rows**: Wind speed, Power, Rotor speed, Rotor torque, Blade pitch, Blade root torque, Average root torque
- **Columns**: WE_Mode=0 (hub) standard, WE_Mode=2 (WSE) standard, kw^2 controller

Each scenario shown with different colors:
- Blue: Baseline
- Orange: Free yaw
- Green: Zero yaw

The average root torque is shown as dashed horizontal lines.

All subplots in each row share the same y-axis limits for easy comparison.

In [ ]:
# Define colors for each scenario
scenario_colors = {
    "Baseline": "C0",  # Blue
    "Free yaw": "C1",  # Orange
    "Zero yaw": "C2",  # Green
}

# Define column mapping for controller modes
controller_columns = {
    ("standard", 0): 0,  # WE_Mode=0 (hub), standard
    ("standard", 2): 1,  # WE_Mode=2 (WSE), standard
    ("kw2", 0): 2,  # WE_Mode=0 (hub), kw2
}

column_titles = {
    0: "WE_Mode=0 (hub)",
    1: "WE_Mode=2 (WSE)",
    2: "kw^2 controller",
}

# Create figure with 7x3 subplots
fig, axes = plt.subplots(
    7,
    3,
    figsize=(
        1.5 * plt.rcParams["figure.figsize"][0],
        3.0 * plt.rcParams["figure.figsize"][1],
    ),
    sharex=True,
)

# Plot wind speed (top row)
for (scenario, we_mode, controller_mode), data in sorted(data_by_scenario.items()):
    col = controller_columns[(controller_mode, we_mode)]
    axes[0, col].plot(
        data["time_rosco"],
        data["wind_speed"],
        color=scenario_colors[scenario],
        linewidth=1.5,
        label=scenario,
        alpha=0.8,
    )

# Set labels and limits for wind speed row
for col in range(3):
    axes[0, col].grid(alpha=0.3)
    axes[0, col].set_title(column_titles[col])
axes[0, 0].set_ylabel("Wind speed (m/s)")
ylims = [axes[0, col].get_ylim() for col in range(3)]
common_ylim = (min(y[0] for y in ylims), max(y[1] for y in ylims))
for col in range(3):
    axes[0, col].set_ylim(common_ylim)

# Plot power (second row)
for (scenario, we_mode, controller_mode), data in sorted(data_by_scenario.items()):
    col = controller_columns[(controller_mode, we_mode)]
    power_mw = data["power"] / 1e3  # from kW to MW
    axes[1, col].plot(
        data["time_openfast"],
        power_mw,
        color=scenario_colors[scenario],
        linewidth=1.5,
        alpha=0.8,
    )

# Set labels and limits for power row
for col in range(3):
    axes[1, col].grid(alpha=0.3)
axes[1, 0].set_ylabel("Power (MW)")
ylims = [axes[1, col].get_ylim() for col in range(3)]
common_ylim = (min(y[0] for y in ylims), max(y[1] for y in ylims))
for col in range(3):
    axes[1, col].set_ylim(common_ylim)

# Plot rotor speed (third row)
for (scenario, we_mode, controller_mode), data in sorted(data_by_scenario.items()):
    col = controller_columns[(controller_mode, we_mode)]
    # Convert rotor speed from rad/s to rpm
    rotor_speed_rpm = data["rotor_speed"]  # already in rpm
    rotor_speed_ref_rpm = data["rotor_speed_ref"] * 60 / (2 * np.pi)

    # Plot actual rotor speed
    axes[2, col].plot(
        data["time_openfast"],
        rotor_speed_rpm,
        color=scenario_colors[scenario],
        linewidth=1.5,
        alpha=0.8,
    )

    # Plot reference rotor speed (thinner, lighter)
    axes[2, col].plot(
        data["time_rosco"],
        rotor_speed_ref_rpm,
        color=scenario_colors[scenario],
        linewidth=0.8,
        alpha=0.4,
    )

# Set labels and limits for rotor speed row
for col in range(3):
    axes[2, col].grid(alpha=0.3)
axes[2, 0].set_ylabel("Rotor speed (rpm)")
ylims = [axes[2, col].get_ylim() for col in range(3)]
common_ylim = (min(y[0] for y in ylims), max(y[1] for y in ylims))
for col in range(3):
    axes[2, col].set_ylim(common_ylim)

# Plot rotor torque (fourth row)
for (scenario, we_mode, controller_mode), data in sorted(data_by_scenario.items()):
    col = controller_columns[(controller_mode, we_mode)]
    # Convert torque from Nm to kNm
    rotor_torque_knm = data["rotor_torque"] / 1e3
    axes[3, col].plot(
        data["time_openfast"],
        rotor_torque_knm,
        color=scenario_colors[scenario],
        linewidth=1.5,
        alpha=0.8,
    )

# Set labels and limits for rotor torque row
for col in range(3):
    axes[3, col].grid(alpha=0.3)
axes[3, 0].set_ylabel("Rotor torque (kNm)")
ylims = [axes[3, col].get_ylim() for col in range(3)]
common_ylim = (min(y[0] for y in ylims), max(y[1] for y in ylims))
for col in range(3):
    axes[3, col].set_ylim(common_ylim)

# Plot blade pitch (fifth row)
for (scenario, we_mode, controller_mode), data in sorted(data_by_scenario.items()):
    col = controller_columns[(controller_mode, we_mode)]
    axes[4, col].plot(
        data["time_openfast"],
        data["blade_pitch"],
        color=scenario_colors[scenario],
        linewidth=1.5,
        alpha=0.8,
    )

# Set labels and limits for blade pitch row
for col in range(3):
    axes[4, col].grid(alpha=0.3)
axes[4, 0].set_ylabel("Blade pitch (deg)")
ylims = [axes[4, col].get_ylim() for col in range(3)]
common_ylim = (min(y[0] for y in ylims), max(y[1] for y in ylims))
for col in range(3):
    axes[4, col].set_ylim(common_ylim)

# Plot blade root torque (sixth row)
for (scenario, we_mode, controller_mode), data in sorted(data_by_scenario.items()):
    col = controller_columns[(controller_mode, we_mode)]
    # Convert root torque from Nm to kNm
    root_torque_knm = data["root_torque"] / 1e3
    axes[5, col].plot(
        data["time_openfast"],
        root_torque_knm,
        color=scenario_colors[scenario],
        linewidth=1.5,
        alpha=0.8,
    )

# Set labels and limits for blade root torque row
for col in range(3):
    axes[5, col].grid(alpha=0.3)
axes[5, 0].set_ylabel("Root torque (kNm)")
ylims = [axes[5, col].get_ylim() for col in range(3)]
common_ylim = (min(y[0] for y in ylims), max(y[1] for y in ylims))
for col in range(3):
    axes[5, col].set_ylim(common_ylim)

# Plot average blade root torque (seventh row)
for (scenario, we_mode, controller_mode), data in sorted(data_by_scenario.items()):
    col = controller_columns[(controller_mode, we_mode)]
    # Convert root torque from Nm to kNm and compute mean
    root_torque_knm = data["root_torque"] / 1e3
    avg_root_torque = np.mean(root_torque_knm)
    axes[6, col].axhline(
        y=avg_root_torque,
        color=scenario_colors[scenario],
        linestyle="--",
        linewidth=1.5,
        alpha=0.8,
    )

# Set labels and limits for average root torque row
for col in range(3):
    axes[6, col].grid(alpha=0.3)
    axes[6, col].set_xlabel("Time (s)")
axes[6, 0].set_ylabel("Avg root torque (kNm)")
ylims = [axes[6, col].get_ylim() for col in range(3)]
common_ylim = (min(y[0] for y in ylims), max(y[1] for y in ylims))
for col in range(3):
    axes[6, col].set_ylim(common_ylim)

# Add legend for scenarios (only in top-right subplot)
axes[0, 2].legend(loc="best")

plt.tight_layout()
plt.show()

## Root Torque Detailed Comparison

Larger plot focusing on blade root torque across the three controller modes.

In [ ]:
# Create larger figure with 1x3 subplots for root torque
fig_root, axes_root = plt.subplots(
    3,
    1,
    figsize=(
        1 * plt.rcParams["figure.figsize"][0],
        3 * plt.rcParams["figure.figsize"][1],
    ),
    sharex=True,
    sharey=True,
)

# Plot blade root torque for each controller mode
for (scenario, we_mode, controller_mode), data in sorted(data_by_scenario.items()):
    col = controller_columns[(controller_mode, we_mode)]
    # Convert root torque from Nm to kNm
    root_torque_knm = data["root_torque"] / 1e3
    axes_root[col].plot(
        data["time_openfast"],
        root_torque_knm,
        color=scenario_colors[scenario],
        linewidth=1.5,
        label=scenario,
        alpha=0.8,
    )

# Set labels and titles for each subplot
for col in range(3):
    axes_root[col].grid(alpha=0.3)
    axes_root[col].set_title(column_titles[col])
    axes_root[col].set_xlabel("Time (s)")
axes_root[0].set_ylabel("Root torque (kNm)")

# Add legend to first subplot
axes_root[0].legend(loc="best")

plt.tight_layout()
plt.show()

## Journal plots

In [ ]:
from pathlib import Path

# Apply the journal style.
plt.style.use("journal.mplstyle")

FIGURE_DIR = Path("../figures")
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def extract_last_period(azimuth, *signals):
    """Extract signals over the last complete rotor revolution using azimuth wraps."""
    daz = np.diff(azimuth)
    # A wrap-around is where the azimuth resets from ~360 back to ~0
    wrap_indices = np.where(daz < -180)[0]
    if len(wrap_indices) >= 2:
        start_idx = wrap_indices[-2] + 1
        end_idx = wrap_indices[-1] + 1
    elif len(wrap_indices) == 1:
        start_idx = 0
        end_idx = wrap_indices[-1] + 1
    else:
        start_idx = 0
        end_idx = len(azimuth)
    return azimuth[start_idx:end_idx], [s[start_idx:end_idx] for s in signals]


# Left column: wind speed estimator (WE_Mode=2), right column: hub height (WE_Mode=0)
journal_col_map = {
    ("standard", 2): 0,
    ("standard", 0): 1,
}
journal_col_titles = ["Wind speed estimator", "Hub height wind speed"]

fig_j, axes_j = plt.subplots(
    4,
    2,
    figsize=(
        plt.rcParams["figure.figsize"][0],
        1.33 * plt.rcParams["figure.figsize"][1],
    ),
    sharex=True,
    sharey="row",
)

for (scenario, we_mode, controller_mode), data in sorted(data_by_scenario.items()):
    # Only plot standard controller, comparing the two wind speed estimation methods
    if controller_mode != "standard":
        continue
    col = journal_col_map[(controller_mode, we_mode)]

    # Interpolate ROSCO wind speed estimate onto the OpenFAST time grid so
    # all signals share the same azimuth axis.
    wind_speed_interp = np.interp(
        data["time_openfast"], data["time_rosco"], data["wind_speed"]
    )

    power_mw = data["power"] / 1e3  # kW → MW
    rotor_speed_rpm = data["rotor_speed"]  # already in rpm
    root_torque_knm = data["root_torque"] / 1e3  # → kNm

    az, (power_p, wind_p, speed_p, torque_p) = extract_last_period(
        data["azimuth"], power_mw, wind_speed_interp, rotor_speed_rpm, root_torque_knm
    )

    print(
        f"For {scenario=}, {we_mode=}, {controller_mode=}: torque mean = {np.mean(torque_p):05.2f}, min = {np.min(torque_p):05.2f}, max = {np.max(torque_p):05.2f}."
    )

    plot_kwargs = dict(
        color=scenario_colors[scenario], linewidth=1.5, label=scenario, alpha=0.8
    )
    axes_j[0, col].plot(az, power_p, **plot_kwargs)
    axes_j[1, col].plot(az, wind_p, **plot_kwargs)
    axes_j[2, col].plot(az, speed_p, **plot_kwargs)
    axes_j[3, col].plot(az, torque_p, **plot_kwargs)

# Set column titles above top row
for col in range(2):
    axes_j[0, col].set_title(journal_col_titles[col])

# Set row y-labels and grid
row_ylabels = [
    "Power (MW)",
    "Estimated wind speed (m/s)",
    "Rotor speed (rpm)",
    "Blade 1 torque (kNm)",
]
for row, ylabel in enumerate(row_ylabels):
    axes_j[row, 0].set_ylabel(ylabel, rotation=0, horizontalalignment="right")
    for col in range(2):
        axes_j[row, col].grid(alpha=0.3)

# x-label on bottom row only
for col in range(2):
    axes_j[3, col].set_xlabel("Azimuth (deg)")

axes_j[0, 1].legend(loc="upper right")

fig_j.savefig(FIGURE_DIR / "wse_hub_power_difference.pdf")
plt.show()